In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoTokenizer, AutoModel
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader, TensorDataset
from sklearn.decomposition import NMF, PCA
from scipy.sparse import hstack

In [2]:
df = pd.read_csv('F:/mental health projct of jp/suicide paper/dataset/2_types_filtered_features_3lakh.csv')
df.head()

,text,Label,filtered_text_eng,tokenized_text,Preprocessed_Text,filtered_tokenized_words
0,"I'm done with it all. Any tips?First of all, i...",2,done with it Any of if going to comment get or...,"['done', 'go', 'comment', 'get', 'plea', 'unde...",done go comment get plea understand want hear ...,"['kill', 'pain', 'sorri', 'hate']"
1,i 20m was a problem child when grow up i frequ...,1,i was a problem child when grow up i frequent ...,"['problem', 'child', 'grow', 'frequent', 'got'...",problem child grow frequent got troubl junior ...,"['ill', 'awkward', 'steal', 'silenc', 'wrong',..."
2,I officially hate my school We have to start p...,0,I officially hate my school We have to start o...,"['offici', 'hate', 'school', 'start', 'enter',...",offici hate school start enter caus fix,"['offici', 'hate', 'school', 'start', 'enter',..."
3,but onc again depress love to take that from m...,1,but onc again depress love to take that from m...,"['onc', 'depress', 'love', 'take', 'angri', 'w...",onc depress love take angri want cri wont come...,"['cri', 'angri', 'damn', 'tire', 'depress']"
4,"Starting today, I'm going to attempt to fix my...",0,Starting going to attempt to fix my sleep sche...,"['start', 'go', 'attempt', 'fix', 'sleep', 'sc...",start go attempt fix sleep schedul school coup...,"['start', 'go', 'attempt', 'fix', 'sleep', 'sc..."


In [3]:
df.isnull().sum()

text                        0
Label                       0
filtered_text_eng           0
tokenized_text              0
Preprocessed_Text           0
filtered_tokenized_words    0
dtype: int64

In [4]:
df.shape

(287398, 6)

In [5]:
# BERT
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert_model = AutoModel.from_pretrained("bert-base-uncased")

In [6]:
# ELECTRA
electra_tokenizer = AutoTokenizer.from_pretrained("google/electra-small-discriminator")
electra_model = AutoModel.from_pretrained("google/electra-small-discriminator")

In [7]:
# Initialize GPT-2 tokenizer and model
gpt_tokenizer = AutoTokenizer.from_pretrained("gpt2")
gpt_model = AutoModel.from_pretrained("gpt2")

# Set pad token as eos token for GPT-2
gpt_tokenizer.pad_token = gpt_tokenizer.eos_token

In [8]:
# XLNet
xlnet_tokenizer = AutoTokenizer.from_pretrained("xlnet-base-cased")
xlnet_model = AutoModel.from_pretrained("xlnet-base-cased")

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [10]:
# Move models to GPU
bert_model.to(device)
electra_model.to(device)
gpt_model.to(device)
xlnet_model.to(device)

XLNetModel(
  (word_embedding): Embedding(32000, 768)
  (layer): ModuleList(
    (0-11): 12 x XLNetLayer(
      (rel_attn): XLNetRelativeAttention(
        (layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): XLNetFeedForward(
        (layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (layer_1): Linear(in_features=768, out_features=3072, bias=True)
        (layer_2): Linear(in_features=3072, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (activation_function): GELUActivation()
      )
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (dropout): Dropout(p=0.1, inplace=False)
)

In [11]:
def get_transformer_embeddings(texts, tokenizer, model, device='cpu', batch_size=16):
    embeddings = []
    model.to(device)  # Move model to device (CPU or GPU)
    model.eval()  # Set model to evaluation mode

    with torch.no_grad():
        # Process texts in batches
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]
            inputs = tokenizer(batch_texts, return_tensors='pt', truncation=True, padding=True, max_length=128)
            input_ids = inputs['input_ids'].to(device)
            attention_mask = inputs['attention_mask'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            last_hidden_states = outputs.last_hidden_state
            batch_embeddings = last_hidden_states.mean(dim=1).cpu().numpy()
            embeddings.append(batch_embeddings)

    return np.vstack(embeddings)

In [12]:
X_bert = get_transformer_embeddings(df['Preprocessed_Text'].tolist(), bert_tokenizer, bert_model, device=device)

In [13]:
X_electra = get_transformer_embeddings(df['Preprocessed_Text'].tolist(), electra_tokenizer, electra_model, device=device)

In [14]:
X_gpt = get_transformer_embeddings(df['Preprocessed_Text'].tolist(), gpt_tokenizer, gpt_model, device=device)

In [15]:
X_xlnet = get_transformer_embeddings(df['Preprocessed_Text'].tolist(), xlnet_tokenizer, xlnet_model, device=device)

In [18]:
# Load GloVe Embeddings
glove_embeddings = {}
glove_file_path = "F:/mental health projct of jp/glove.6B.100d.txt"
with open(glove_file_path, encoding="utf-8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype="float32")
        glove_embeddings[word] = vector

# Word Embeddings using GloVe
def get_glove_vector(tokens, glove_embeddings, dim=100):
    embeddings = [glove_embeddings.get(word, np.zeros(dim)) for word in tokens]
    return np.mean(embeddings, axis=0) if embeddings else np.zeros(dim)

In [19]:
X_glove = np.array([get_glove_vector(tokens, glove_embeddings) for tokens in df["filtered_tokenized_words"]])

In [20]:
# TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=100, ngram_range=(1, 3))
X_tfidf = tfidf_vectorizer.fit_transform(df["filtered_tokenized_words"]).toarray()

In [21]:
from sklearn.feature_extraction.text import CountVectorizer

# Count Vectorizer with custom tokenizer and preprocessor
count_vectorizer = CountVectorizer(
    max_features=100,           # Limit to 100 features
    tokenizer=lambda x: x,      
    preprocessor=lambda x: x   
)

# Transform the data
X_count = count_vectorizer.fit_transform(df['filtered_tokenized_words']).toarray()

F:\JP database project\pythonProject\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [22]:
X_combined = np.hstack((X_glove, X_tfidf, X_count, X_bert, X_electra, X_gpt, X_xlnet))

In [23]:
# Convert combined features to tensor
X_final = torch.tensor(X_combined, dtype=torch.float32)

In [24]:
# Convert tensor to NumPy array
X_final_np = X_final.numpy()

# Create a DataFrame from the NumPy array
df_final = pd.DataFrame(X_final_np)

# Save the DataFrame to a CSV file 
df_final.to_csv('F:/mental health projct of jp/suicide paper/dataset/2_types_all_features.csv', index=False)

In [25]:
df_final.head(20)

,0,1,2,3,4,5,6,7,8,9,...,2781,2782,2783,2784,2785,2786,2787,2788,2789,2790
0,-0.311691,0.207526,0.294796,-0.370256,-0.463946,0.004523,0.103642,0.023245,-0.273210,-0.043930,...,1.174970,-0.131030,0.581713,1.317829,0.545300,-4.912285,2.002302,-2.429308,2.313626,0.068317
1,-0.314458,0.200302,0.248307,-0.337924,-0.354438,-0.029178,0.120759,0.028209,-0.381361,-0.010982,...,-0.170022,-1.057124,0.506787,0.481089,-0.443753,-1.015673,0.295432,-1.521313,-0.511028,-0.270724
2,-0.258834,0.199409,0.255281,-0.329083,-0.421285,-0.008494,0.122459,0.000529,-0.345936,-0.007731,...,-0.475023,-2.049421,-2.095686,-2.270150,2.747774,-4.004583,-1.818186,-3.040991,-0.021306,1.718746
3,-0.348873,0.164237,0.313819,-0.395596,-0.340182,-0.008542,0.077583,-0.002947,-0.369009,0.005620,...,1.116820,0.496508,2.048395,-0.856290,0.582634,-1.573221,0.650254,-1.483610,0.080408,-0.893238
4,-0.258569,0.201524,0.242732,-0.321395,-0.432100,-0.044947,0.111697,0.009705,-0.350573,-0.033735,...,-0.861691,-1.552952,-0.070507,-2.612564,0.754670,-5.594344,3.663493,-0.440785,1.437596,-0.616005
5,-0.277951,0.160797,0.290795,-0.305743,-0.439769,-0.011427,0.093815,0.002534,-0.280741,-0.050374,...,0.328833,-0.130070,-0.398651,0.483433,0.527054,-2.224771,0.269342,-2.088394,1.147086,-1.117992
6,-0.328637,0.298792,0.285169,-0.268216,-0.442173,-0.005997,0.121161,0.091326,-0.473951,0.054511,...,-0.195480,-1.666454,0.691747,0.216133,0.736728,-3.127229,-0.526728,-1.756149,-0.267481,-0.249448
7,-0.316708,0.169958,0.213350,-0.366424,-0.353601,0.015562,0.124488,-0.027506,-0.358802,-0.046981,...,1.368074,-1.310359,-2.792422,-4.041125,0.888524,-6.746804,-0.641374,0.891108,2.391107,-0.535528
8,-0.280333,0.310111,0.193596,-0.258668,-0.526899,0.025660,0.173693,0.014277,-0.430222,-0.034677,...,1.064472,-0.440751,0.247901,-1.815318,0.336435,-4.959441,1.330516,-3.148606,-0.100648,0.013222
9,-0.281939,0.166514,0.235172,-0.350586,-0.360470,-0.022592,0.132992,0.024310,-0.314069,-0.023638,...,1.688187,-1.166405,1.806003,0.238366,-1.340906,-3.654136,2.615344,-1.513549,2.204822,-1.370530


In [26]:
# Convert combined features to tensor
X_bert_final = torch.tensor(X_bert, dtype=torch.float32)

# Convert tensor to NumPy array
X_bert_final_np = X_bert_final.numpy()

# Create a DataFrame from the NumPy array
df_final_X_bert = pd.DataFrame(X_bert_final_np)

# Save the DataFrame to a CSV file 
df_final_X_bert.to_csv('F:/mental health projct of jp/suicide paper/dataset/2_types_bert.csv', index=False)
df_final_X_bert.head(20)

,0,1,2,3,4,5,6,7,8,9,...,758,759,760,761,762,763,764,765,766,767
0,0.039339,0.027318,0.542960,0.067279,0.035454,0.057449,0.192009,0.011376,0.099093,-0.457100,...,-0.026904,-0.287610,-0.026079,-0.081054,-0.172036,0.204499,-0.023147,-0.176354,-0.130857,0.055808
1,-0.491383,-0.116573,0.474933,-0.000260,0.101972,-0.092985,0.298460,0.312335,0.049858,-0.411412,...,-0.029965,-0.137379,0.017629,0.033461,-0.035369,0.011849,-0.204691,-0.353506,-0.099069,0.056156
2,0.004749,-0.169299,0.094293,-0.033749,0.114623,0.008532,0.428238,0.353712,0.106186,-0.518744,...,0.207332,-0.199377,0.141076,0.200267,-0.090276,-0.081950,0.014670,-0.250804,-0.021329,0.226559
3,-0.203363,-0.051969,0.642217,-0.132124,-0.112664,-0.007807,0.282154,0.017729,0.194809,-0.252398,...,-0.049852,-0.410159,-0.010218,0.092821,-0.256525,0.253893,-0.082146,-0.270716,-0.134515,0.038669
4,-0.062877,-0.319566,0.300077,0.008816,0.305638,-0.206011,0.217094,-0.085043,0.095858,-0.175442,...,0.122658,-0.236464,0.093678,-0.079467,-0.015312,0.244646,-0.044545,-0.148862,-0.062770,-0.102614
5,-0.146715,0.028845,0.487284,0.010829,-0.097013,-0.161657,0.276186,0.372388,0.296024,-0.335906,...,0.313471,-0.226476,-0.075263,0.129500,-0.141698,0.119048,-0.013919,-0.285261,-0.007778,0.051887
6,-0.229220,-0.071243,0.659669,0.057474,0.180020,-0.169151,0.083695,0.090853,0.136538,-0.457812,...,0.131409,-0.367416,0.155207,-0.029633,0.013475,0.161376,-0.066659,-0.361590,-0.200353,0.137712
7,-0.024275,0.140617,0.210345,-0.032039,-0.129329,0.106952,0.298608,0.525600,0.201475,-0.591593,...,0.044255,-0.403285,0.117266,0.106771,-0.120464,-0.056509,0.219332,0.027429,0.399939,-0.029759
8,0.101470,-0.151218,0.309608,0.149115,0.133244,-0.149087,0.254840,0.013539,0.158101,-0.155651,...,0.188079,-0.379466,0.075879,-0.038100,0.070519,0.288737,-0.027767,-0.299967,-0.065828,0.114659
9,0.079476,-0.306718,0.467059,0.079831,0.261917,-0.038099,0.228091,0.080303,0.087497,-0.394975,...,0.145723,-0.326567,0.310158,-0.022558,-0.140809,0.093799,-0.190606,-0.129748,-0.178369,-0.067854


In [27]:
# Convert combined features to tensor
X_electra_final = torch.tensor(X_electra, dtype=torch.float32)

# Convert tensor to NumPy array
X_electra_final_np = X_electra_final.numpy()

# Create a DataFrame from the NumPy array
df_final_X_electra = pd.DataFrame(X_electra_final_np)

# Save the DataFrame to a CSV file 
df_final_X_electra.to_csv('F:/mental health projct of jp/suicide paper/dataset/2_types_electra.csv', index=False)
df_final_X_electra.head(20)

,0,1,2,3,4,5,6,7,8,9,...,246,247,248,249,250,251,252,253,254,255
0,0.599472,0.439625,0.020266,0.138930,-0.609429,-0.539497,-0.403593,-0.137966,0.020127,0.690899,...,0.141068,0.108030,-0.510015,-1.147126,-0.002389,-0.206351,1.127947,-0.253208,-0.523452,-0.328024
1,0.250966,0.313902,-0.319763,0.109657,-0.361788,-0.077295,0.165057,0.165287,0.238635,1.128709,...,0.042352,0.410725,-0.239818,-0.738911,-0.009571,-0.778596,0.321679,-0.342413,-0.209100,-0.151962
2,0.798442,0.137798,0.217636,0.615689,-0.422624,-0.667775,-0.803210,-0.369768,-0.318564,-0.041989,...,0.041961,-0.172283,-0.642361,-0.670355,-0.119998,0.519885,1.531320,-0.087807,-0.414092,-0.394284
3,0.694335,0.538833,0.078782,0.329489,-0.570069,-0.392096,-0.639536,-0.064476,-0.135712,0.484850,...,0.131051,-0.011844,-0.597980,-0.633558,0.011597,0.206809,1.234858,-0.258696,-0.551979,-0.394734
4,0.753654,0.343877,0.177863,0.563893,-0.583629,-0.589239,-0.902104,-0.213953,-0.210492,0.123507,...,0.040914,-0.103076,-0.607364,-0.749816,0.032885,0.400656,1.501542,-0.132509,-0.436215,-0.385705
5,0.556395,0.336273,-0.044140,0.330744,-0.528795,-0.410875,-0.402967,-0.000822,-0.132016,0.692666,...,0.120789,0.038540,-0.543584,-0.826162,0.019728,-0.075651,1.121055,-0.201159,-0.669734,-0.347811
6,0.606831,0.416997,-0.089709,0.383873,-0.437087,-0.447979,-0.397252,0.010724,-0.038786,0.519137,...,0.102338,0.211360,-0.472772,-0.861665,-0.001110,-0.238651,0.985930,-0.143282,-0.557462,-0.206413
7,0.889869,0.200161,0.211762,0.602617,-0.216460,-0.731466,-0.723430,-0.237504,-0.379962,-0.128080,...,-0.015177,-0.200958,-0.557887,-0.615360,-0.241240,0.829681,1.098043,-0.295733,-0.741909,-0.611906
8,0.490453,0.138842,0.093108,0.606177,-0.387260,-0.540420,-0.358276,-0.192229,-0.303791,0.096349,...,0.186031,-0.042873,-0.751456,-0.775492,-0.183552,0.389557,1.154379,-0.125627,-0.509174,-0.514307
9,0.506974,0.091086,0.002913,0.419425,-0.413436,-0.423930,-0.367566,0.084806,-0.206073,0.511356,...,0.238478,0.067842,-0.698654,-0.871200,-0.124419,-0.003843,1.044604,-0.006547,-0.466472,-0.294304


In [28]:
# Convert combined features to tensor
X_gpt_final = torch.tensor(X_gpt, dtype=torch.float32)

# Convert tensor to NumPy array
X_gpt_final_np = X_gpt_final.numpy()

# Create a DataFrame from the NumPy array
df_final_X_gpt = pd.DataFrame(X_gpt_final_np)

# Save the DataFrame to a CSV file 
df_final_X_gpt.to_csv('F:/mental health projct of jp/suicide paper/dataset/2_types_gpt.csv', index=False)
df_final_X_gpt.head(20)

,0,1,2,3,4,5,6,7,8,9,...,758,759,760,761,762,763,764,765,766,767
0,-0.175703,-0.128528,-0.084935,-0.095990,0.017343,0.040334,4.020577,-0.036600,0.177849,0.013525,...,-0.195830,0.354232,-0.207118,-0.020549,2.571847,0.269671,0.355543,-0.200342,0.132587,0.053167
1,-0.053350,-0.158133,-0.083422,0.027374,-0.146509,0.105708,-1.912807,-0.095139,0.337502,0.320858,...,-0.540632,0.145271,-0.082381,0.016155,4.346670,0.138111,0.160342,-0.194968,0.396577,0.260527
2,-0.104494,-0.081889,0.118526,-0.011474,0.102028,-0.325892,4.923399,-0.233921,0.015988,0.042901,...,-0.197509,0.139917,-0.156645,-0.157115,0.840378,0.213294,0.154509,-0.011298,0.103850,-0.031862
3,-0.161296,-0.008129,-0.258956,-0.103768,0.145393,0.025114,13.907271,0.098031,-0.028736,-0.069874,...,-0.077221,0.282741,-0.155734,-0.002357,2.100598,0.299896,0.207297,-0.125741,-0.037304,-0.047917
4,-0.001621,-0.146236,0.058283,-0.089719,0.079217,-0.047845,10.313022,-0.054384,0.008916,-0.126622,...,-0.053217,0.129640,-0.228580,-0.135887,0.790294,0.242170,0.237762,-0.040008,-0.005287,-0.090784
5,-0.230178,0.059749,-0.237079,-0.141400,0.097137,0.059281,11.449949,0.014305,0.171883,0.031766,...,-0.110786,0.379071,-0.150732,0.045897,2.436574,0.209442,0.264724,-0.183146,0.070527,-0.093800
6,-0.150603,0.035567,-0.245175,-0.099363,0.055164,-0.067794,9.465238,0.052797,0.045800,0.070554,...,-0.065945,0.544944,-0.012305,-0.177044,3.173671,0.233340,0.136867,-0.149710,0.054362,0.020757
7,-0.245142,0.066866,-0.160787,-0.152481,0.051272,-0.041539,16.373730,0.082881,-0.049279,-0.152321,...,-0.017368,0.272946,-0.289343,-0.009536,1.389789,0.173425,0.198056,-0.010321,-0.249162,-0.197115
8,-0.084476,0.074708,-0.167517,-0.167617,0.092356,-0.011225,13.419622,0.023389,0.009952,0.063642,...,-0.079386,0.359535,-0.133965,-0.102594,1.909375,0.318847,0.237308,-0.090948,-0.061338,-0.134615
9,-0.312247,-0.128049,-0.151016,0.009313,0.109228,0.071397,7.738469,0.015650,0.046567,0.027857,...,-0.011443,0.334339,-0.153144,-0.099489,2.088679,0.279512,0.231282,0.004075,0.031337,-0.026659


In [29]:
# Convert combined features to tensor
X_xlnet_final = torch.tensor(X_xlnet, dtype=torch.float32)

# Convert tensor to NumPy array
X_xlnet_final_np = X_xlnet_final.numpy()

# Create a DataFrame from the NumPy array
df_final_X_xlnet = pd.DataFrame(X_xlnet_final_np)

# Save the DataFrame to a CSV file 
df_final_X_xlnet.to_csv('F:/mental health projct of jp/suicide paper/dataset/2_types_xlnet.csv', index=False)
df_final_X_xlnet.head(20)

,0,1,2,3,4,5,6,7,8,9,...,758,759,760,761,762,763,764,765,766,767
0,0.759139,-0.219624,-2.537084,1.348347,0.156367,-1.809295,0.046765,1.175759,-4.395433,0.086587,...,1.174970,-0.131030,0.581713,1.317829,0.545300,-4.912285,2.002302,-2.429308,2.313626,0.068317
1,-0.061249,-0.591833,-1.094049,0.190955,-0.056015,-0.131815,0.305452,0.940081,-0.292011,-0.022841,...,-0.170022,-1.057124,0.506787,0.481089,-0.443753,-1.015673,0.295432,-1.521313,-0.511028,-0.270724
2,1.201066,-2.194218,-0.574711,-1.052986,-3.301692,-5.273767,-0.233267,4.624640,0.466496,1.650364,...,-0.475023,-2.049421,-2.095686,-2.270150,2.747774,-4.004583,-1.818186,-3.040991,-0.021306,1.718746
3,-0.163431,0.753550,-1.646211,0.441859,-1.658335,0.302772,-0.637827,2.469981,-1.733055,0.607658,...,1.116820,0.496508,2.048395,-0.856290,0.582634,-1.573221,0.650254,-1.483610,0.080408,-0.893238
4,-0.700489,-0.445789,-1.847687,2.117341,-1.111654,-1.074585,-2.676440,1.812639,-1.588404,-0.335439,...,-0.861691,-1.552952,-0.070507,-2.612564,0.754670,-5.594344,3.663493,-0.440785,1.437596,-0.616005
5,-1.118942,-0.317036,-1.448065,-0.214859,-0.322030,-2.265769,-0.303564,2.556220,-2.614882,1.630911,...,0.328833,-0.130070,-0.398651,0.483433,0.527054,-2.224771,0.269342,-2.088394,1.147086,-1.117992
6,1.429429,-0.042022,-1.947446,0.460861,-2.092763,-2.315769,-1.036956,1.579914,-1.373634,0.517815,...,-0.195480,-1.666454,0.691747,0.216133,0.736728,-3.127229,-0.526728,-1.756149,-0.267481,-0.249448
7,1.196037,-0.799787,-1.495339,-3.110963,-1.556353,-4.769216,-1.586578,0.719475,-1.252651,3.661518,...,1.368074,-1.310359,-2.792422,-4.041125,0.888524,-6.746804,-0.641374,0.891108,2.391107,-0.535528
8,0.293644,-0.243827,-2.146852,1.271080,-0.764018,-0.741198,0.208969,1.083510,-2.964894,1.536112,...,1.064472,-0.440751,0.247901,-1.815318,0.336435,-4.959441,1.330516,-3.148606,-0.100648,0.013222
9,1.239203,-1.966399,-1.242372,-0.064252,-0.578578,-1.580602,-0.196728,1.028078,-2.087779,-1.607386,...,1.688187,-1.166405,1.806003,0.238366,-1.340906,-3.654136,2.615344,-1.513549,2.204822,-1.370530


In [30]:
# Convert combined features to tensor
X_glove_final = torch.tensor(X_glove, dtype=torch.float32)

# Convert tensor to NumPy array
X_glove_final_np = X_glove_final.numpy()

# Create a DataFrame from the NumPy array
df_final_X_glove = pd.DataFrame(X_glove_final_np)

# Save the DataFrame to a CSV file 
df_final_X_glove.to_csv('F:/mental health projct of jp/suicide paper/dataset/2_types_glove.csv', index=False)
df_final_X_glove.head(20)

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,-0.311691,0.207526,0.294796,-0.370256,-0.463946,0.004523,0.103642,0.023245,-0.273210,-0.043930,...,-0.301235,0.074111,0.174775,0.388905,-0.109968,-0.105841,-0.103873,-0.672314,0.437760,-0.023661
1,-0.314458,0.200302,0.248307,-0.337924,-0.354438,-0.029178,0.120759,0.028209,-0.381361,-0.010982,...,-0.338166,0.154628,0.229163,0.413844,-0.109221,-0.065236,-0.088432,-0.681907,0.429138,-0.123187
2,-0.258834,0.199409,0.255281,-0.329083,-0.421285,-0.008494,0.122459,0.000529,-0.345936,-0.007731,...,-0.265862,0.177597,0.220011,0.334313,-0.126213,-0.064588,-0.089246,-0.689013,0.371360,-0.098661
3,-0.348873,0.164237,0.313819,-0.395596,-0.340182,-0.008542,0.077583,-0.002947,-0.369009,0.005620,...,-0.405243,0.149122,0.212679,0.509583,-0.151348,-0.024454,-0.106332,-0.647379,0.424145,-0.109436
4,-0.258569,0.201524,0.242732,-0.321395,-0.432100,-0.044947,0.111697,0.009705,-0.350573,-0.033735,...,-0.280412,0.195986,0.226042,0.345849,-0.083964,-0.068125,-0.107052,-0.717904,0.402261,-0.075529
5,-0.277951,0.160797,0.290795,-0.305743,-0.439769,-0.011427,0.093815,0.002534,-0.280741,-0.050374,...,-0.327521,0.114421,0.194026,0.413322,-0.150393,-0.111980,-0.075506,-0.667942,0.366836,-0.083949
6,-0.328637,0.298792,0.285169,-0.268216,-0.442173,-0.005997,0.121161,0.091326,-0.473951,0.054511,...,-0.352277,0.201437,0.158911,0.447826,-0.116410,-0.075476,-0.036333,-0.650323,0.345833,-0.047207
7,-0.316708,0.169958,0.213350,-0.366424,-0.353601,0.015562,0.124488,-0.027506,-0.358802,-0.046981,...,-0.347679,0.206957,0.314258,0.413638,-0.072829,-0.022229,-0.079478,-0.720027,0.386465,-0.118947
8,-0.280333,0.310111,0.193596,-0.258668,-0.526899,0.025660,0.173693,0.014277,-0.430222,-0.034677,...,-0.348943,0.156242,0.244309,0.327531,-0.071410,-0.051304,-0.031560,-0.592369,0.306079,-0.105363
9,-0.281939,0.166514,0.235172,-0.350586,-0.360470,-0.022592,0.132992,0.024310,-0.314069,-0.023638,...,-0.315188,0.177110,0.240630,0.409813,-0.102384,-0.043537,-0.118887,-0.713127,0.402605,-0.095379


In [31]:
# Convert combined features to tensor
X_tfidf_final = torch.tensor(X_tfidf, dtype=torch.float32)

# Convert tensor to NumPy array
X_tfidf_final_np = X_tfidf_final.numpy()

# Create a DataFrame from the NumPy array
df_final_X_tfidf = pd.DataFrame(X_tfidf_final_np)

# Save the DataFrame to a CSV file 
df_final_X_tfidf.to_csv('F:/mental health projct of jp/suicide paper/dataset/2_types_tfidf.csv', index=False)
df_final_X_tfidf.head(20)

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000
1,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.336096,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.409004,0.0,0.365831
2,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000
3,0.000000,0.000000,0.000000,0.0,0.639717,0.0,0.0,0.000000,0.0,0.482207,...,0.490736,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000
4,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000
5,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000
6,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000
7,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000
8,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000
9,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000


In [32]:
# Convert combined features to tensor
X_count_final = torch.tensor(X_count, dtype=torch.float32)

# Convert tensor to NumPy array
X_count_final_np = X_count_final.numpy()

# Create a DataFrame from the NumPy array
df_final_X_count = pd.DataFrame(X_count_final_np)

# Save the DataFrame to a CSV file 
df_final_X_count.to_csv('F:/mental health projct of jp/suicide paper/dataset/2_types_count.csv', index=False)
df_final_X_count.head(20)

,0,1,2,3,4,5,6,7,8,9,...,21,22,23,24,25,26,27,28,29,30
0,3.0,8.0,3.0,1.0,1.0,2.0,0.0,0.0,0.0,1.0,...,0.0,2.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,18.0,38.0,18.0,1.0,1.0,7.0,4.0,5.0,5.0,8.0,...,0.0,11.0,5.0,9.0,4.0,0.0,4.0,0.0,0.0,0.0
2,6.0,14.0,6.0,1.0,1.0,3.0,0.0,3.0,0.0,3.0,...,0.0,2.0,3.0,4.0,1.0,0.0,0.0,1.0,0.0,0.0
3,4.0,10.0,4.0,1.0,1.0,2.0,0.0,1.0,2.0,3.0,...,0.0,4.0,2.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,13.0,28.0,13.0,1.0,1.0,3.0,0.0,3.0,1.0,9.0,...,0.0,3.0,7.0,9.0,2.0,0.0,0.0,1.0,0.0,0.0
5,5.0,12.0,5.0,1.0,1.0,1.0,0.0,1.0,1.0,2.0,...,0.0,2.0,1.0,3.0,3.0,0.0,0.0,0.0,0.0,0.0
6,2.0,6.0,2.0,1.0,1.0,3.0,0.0,0.0,2.0,1.0,...,0.0,2.0,3.0,2.0,1.0,0.0,0.0,0.0,0.0,0.0
7,6.0,14.0,6.0,1.0,1.0,3.0,1.0,2.0,1.0,7.0,...,0.0,3.0,2.0,2.0,0.0,1.0,2.0,0.0,0.0,0.0
8,1.0,4.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,...,0.0,0.0,0.0,2.0,1.0,0.0,0.0,0.0,0.0,0.0
9,37.0,76.0,37.0,1.0,1.0,15.0,4.0,12.0,10.0,27.0,...,0.0,10.0,11.0,9.0,5.0,1.0,1.0,0.0,2.0,0.0


In [37]:
print(df_final.shape)
print(df_final_X_bert.shape)
print(df_final_X_electra.shape)
print(df_final_X_gpt.shape)
print(df_final_X_xlnet.shape)
print(df_final_X_glove.shape)
print(df_final_X_tfidf.shape)
print(df_final_X_count.shape)

(287398, 2791)
(287398, 768)
(287398, 256)
(287398, 768)
(287398, 768)
(287398, 100)
(287398, 100)
(287398, 31)
